# Reverse Engineering: Draeger PulmoVista500 `.bin` Format

**Device**: Draeger PulmoVista500 -- firmware V1.30  
**Implemented in**: `src/fasteit/dtypes.py` → `FRAME_EXT_DTYPE`, `MEDIBUS_BASE_FIELDS` / `MEDIBUS_EXT_FIELDS`

**Frame size**: Dräger PulmoVista can writes **4382-byte frames** (`FRAME_EXT_DTYPE`), regardless of PressurePod connection. `FRAME_BASE_DTYPE` (4358 bytes) is documented in eitprocessing repo from EIT-ALIVE (link below) and probably still used from some firmware.

## Prior art

- **EIDORS** (GPL-3, MATLAB): `eidors_readdata.m` subfunctions `read_draeger_header`/`read_draeger_file`. Used to understand field ordering. No code copied (GPL/Apache-2.0 incompatible).
  https://github.com/eidors3d/eidors-readonly

- **eitprocessing** (Apache-2.0, Python): `datahandling/loading/draeger.py` -- field names and sentinel values. Code studied, not copied. 
  https://github.com/EIT-ALIVE/eitprocessing

  **Neither source provides a written format specification.**

In [1]:
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'fasteit').exists():
            return candidate
    raise FileNotFoundError('Could not locate the fastEIT repository root from the current working directory.')

REPO_ROOT = find_repo_root(Path.cwd())
TEST_FILES = REPO_ROOT / 'src' / 'fasteit' / 'test_files'

# Recordings with PressurePod attached
files_pod = [
    TEST_FILES / 'patient01.bin',
    TEST_FILES / 'patient02.bin',
]
# Recording without PressurePod (01_Healthy_Lung) — also 4382 bytes/frame
NO_POD_FILE = Path('/home/ric/usefullfile/prove_fastEIT/01_Healthy_Lung_01.bin')
NO_POD_ASC  = Path('/home/ric/usefullfile/prove_fastEIT/01_Healthy_Lung_01.asc')

# Keep old variable names as aliases for backward compat with code cells below
files_ext = files_pod
BASE_FILE = NO_POD_FILE
BASE_ASC  = NO_POD_ASC

print('=== Files with PressurePod ===')
for f in files_pod:
    print(f'  {f.stat().st_size/1e6:6.1f} MB  {f.name}')
print('=== File without PressurePod ===')
print(f'  {NO_POD_FILE.stat().st_size/1e6:6.1f} MB  {NO_POD_FILE.name}')

=== Files with PressurePod ===
    50.4 MB  patient01.bin
    53.2 MB  patient02.bin
=== File without PressurePod ===
     4.9 MB  01_Healthy_Lung_01.bin


## Frame size

The `.asc` header says `patient01 → 11500 Images`. If the file is a flat sequence of equal frames, `total_bytes / n_frames` must be an exact integer.

In [11]:
p1 = (50_393_000, 11_500)
for label, (b, fr) in [('patient01', p1)]:
    print(f'{label} : {b:,} / {fr:,} = {b/fr} bytes/frame')
print('-> FRAME_EXT_DTYPE.itemsize = 4382')

patient01 : 50,393,000 / 11,500 = 4382.0 bytes/frame
-> FRAME_EXT_DTYPE.itemsize = 4382


Exact integer `FRAME_EXT_DTYPE` (4382 bytes) confirmed.

`01_Healthy_Lung_01.bin` (no PressurePod) is also 4382 bytes per frame — verified in the no-PressurePod section below.

## Full frame hexdump — frame 0 (4382 bytes)

First frame read raw and printed in xxd format, annotated at field boundaries.
The code cell below emulates in Python the output of:

```bash
hexdump -C -s 0 -n 4382 patient01.bin
```

In [3]:
import numpy as np
from fasteit.dtypes import FRAME_EXT_DTYPE

f = np.memmap(TEST_FILES / 'patient01.bin', dtype=FRAME_EXT_DTYPE, mode='r')
raw = bytes(f[0].tobytes())

FIELDS = [
    (0,    'ts (float64 LE, fraction of day)'),
    (8,    'dummy (4 B)'),
    (12,   'pixels[32,32] (4096 B, float32 LE)'),
    (4108, 'min_max_flag + event_marker + event_text + timing_error'),
    (4150, 'medibus_data[0..51] (208 B = 52 x float32)'),
    (4358, 'medibus_data[52..57] -- PressurePod (24 B)'),
]
field_at_row = {}
for offset, name in FIELDS:
    row = (offset // 16) * 16
    field_at_row.setdefault(row, []).append(
        name if offset % 16 == 0 else f'{name} @ +{offset - row:#x}'
    )

for row in range(0, len(raw), 16):
    if row in field_at_row:
        for ann in field_at_row[row]:
            print(f'<-------- {ann} -------->')
    chunk = raw[row:row+16]
    hex_p = ' '.join(f'{b:02x}' for b in chunk)
    asc_p = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)
    print(f'{row:08x}: {hex_p:<47}  |{asc_p}|')

<-------- ts (float64 LE, fraction of day) -------->
<-------- dummy (4 B) @ +0x8 -------->
<-------- pixels[32,32] (4096 B, float32 LE) @ +0xc -------->
00000000: b5 d6 3e cf 02 3e e8 3f 3e a9 20 b7 00 00 00 00  |..>..>.?>. .....|
00000010: 00 00 00 00 00 00 00 00 e2 bd 92 ba eb 9d bc bc  |................|
00000020: 55 b7 bf bd 91 5d 75 be 9c 44 f4 be 5e 4b 29 bf  |U....]u..D..^K).|
00000030: d3 a0 19 bf 9e 5e b4 be 18 64 d4 bd e2 15 54 be  |.....^...d....T.|
00000040: 02 5b 0a bf fa e7 7b bf 0f e3 a8 bf 02 1a bf bf  |.[....{.........|
00000050: 7e ea c8 bf 52 37 aa bf 79 69 88 bf 3a f3 2b bf  |~...R7..yi..:.+.|
00000060: 16 fb fb be 96 bb 01 bf 3d 91 0c bf 96 2b ef be  |........=....+..|
00000070: 70 7e 9a be 34 8e ad bd 99 a4 3c bc 36 8e 54 3a  |p~..4.....<.6.T:|
00000080: 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00  |................|
00000090: 00 00 00 00 28 7c eb b8 18 7e a9 bc db 77 ac bd  |....(|...~...w..|
000000a0: 67 37 65 be f2 91 eb be 57 86 47 bf 21 1b 68 bf  |g7e...

Reading the hexdump:
- `b5 d6 3e cf 02 3e e8 3f` (bytes 0-7): timestamp, decodes to 18:10:54
- `3e a9 20 b7` (bytes 8-11): dummy field, unused
- bytes 0x000C-0x100B: 1024 float32 pixels — note the zero blocks at row boundaries (outside-chest pixels)
- `9e c9 7f ff` repeating at 0x1036+: sentinel `0xFF7FC99E` = MEDIBUS not connected
- `00 00 7a c4` repeating later: `0xC47A0000` = -1000.0, no breath computed yet / non-BiLevel mode
- last 24 bytes (6 × float32): PHigh, Plow, Tlow, ~Paw, ~Pes, ~Ptp, ~Pgas (EXT-specific fields)

## Timestamp

In [4]:
import struct

ts0 = f[0]['ts']
h, m = int(ts0*24), int(ts0*24*60 % 60)
s = ts0*86400 % 60
print(f'ts[0] = {ts0:.8f}  ->  {h:02d}:{m:02d}:{s:04.1f}  (matches .eit header  18:10:54.015)')

ts = f['ts']
dt = np.diff(ts) * 86400
print(f'frames    {len(ts)}')
print(f'duration  {(ts[-1]-ts[0])*86400:.1f} s  (= {len(ts)} x 0.02 s @ 50 Hz)')
print(f'monotone  {bool(np.all(dt > 0))}')
print(f'dt        {dt.mean():.5f} s / {dt.std():.5f} s std')

ts[0] = 0.75756970  ->  18:10:54.0  (matches .eit header  18:10:54.015)
frames    11500
duration  230.0 s  (= 11500 x 0.02 s @ 50 Hz)
monotone  True
dt        0.02000 s / 0.00000 s std


## Pixels

In [6]:
px = f['pixels']
print(f'shape  {px.shape}')
print(f'min    {px.min():.4f}')
print(f'max   +{px.max():.4f}')
print(f'NaN    {int(np.sum(np.isnan(px)))}')
print(f'Inf    {int(np.sum(np.isinf(px)))}')

shape  (11500, 32, 32)
min    -7.7288
max   +28.5301
NaN    0
Inf    0


32x32 pixels, no NaN/Inf. Corner pixels are 0.0 (outside chest boundary). Values are relative impedance changes (Draeger internal units, not Ohm).

## Breath markers

In [ ]:
flags = f['min_max_flag']
idx_neg = np.where(flags == -1)[0]
idx_pos = np.where(flags == +1)[0]
print(f'min_max_flag first 20: {flags[:20].tolist()}')
print(f'troughs (-1): frames {idx_neg[:5].tolist()}')
print(f'peaks   (+1): frames {idx_pos[:5].tolist()}')
dt_f = idx_neg[1] - idx_neg[0]
print(f'-> ~{dt_f/50:.2f} s/breath  ~{60/(dt_f/50):.0f} bpm')

min_max_flag first 20: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
troughs (-1): frames [10, 127, 241, 357, 523]
peaks   (+1): frames [62, 178, 292, 410, 578]
-> ~2.34 s/breath  ~26 bpm


## MEDIBUS field layout: BASE vs EXT

Both formats share `medibus_data[0..50]` (51 fields, identical). They **diverge at idx 51**:

**BASE format (4358 bytes)** — 52 fields total, `MEDIBUS_BASE_FIELDS`:
| idx | field | notes |
|-----|-------|-------|
| 0-50 | (common fields) | see code cell below |
| **51** | `time_at_low_pressure` (Tlow) | duration at low CPAP in BiLevel mode; `-1000.0` sentinel in conventional ventilation |

**EXT format (4382 bytes)** — 58 fields total, `MEDIBUS_EXT_FIELDS`:
| idx | field | notes |
|-----|-------|-------|
| 0-50 | (common fields) | identical to BASE |
| **51** | `high_pressure` (PHigh) | peak CPAP per breath in BiLevel mode; **different field from BASE idx 51** |
| 52 | `low_pressure` (Plow) | low CPAP pressure in BiLevel mode |
| 53 | `time_at_low_pressure` (Tlow) | same parameter as BASE idx 51, but **shifted by +2** |
| 54 | `airway_pressure_pod` (~Paw) | airway pressure waveform, PressurePod transducer |
| 55 | `esophageal_pressure_pod` (~Pes) | esophageal pressure waveform |
| 56 | `transpulmonary_pressure_pod` (~Ptp) | transpulmonary pressure = Paw − Pes (derived) |
| 57 | `gastric_pressure_pod` (~Pgas) | gastric or auxiliary pressure |


## MEDIBUS sentinels

| hex | float32 | meaning |
| --- | ------- | ------- |
| `0xFF7FC99E` | ~ -3.4e38 | MEDIBUS hardware **not connected** |
| `0xC47A0000` |   -1000.0 | connected, **not calculated yet** |

In [7]:
names = ['airway_pressure','flow','volume','co2_pct','co2_kpa','co2_mmhg']
for i, name in enumerate(names):
    v = f[0]['medibus_data'][i]
    print(f'idx {i:2d}  {name:<20}  {float(v):>12.3e}   {int(v.view(np.uint32)):#010x}')
print()
for i in [6, 7]:
    v = f[0]['medibus_data'][i]
    print(f'idx {i:2d}  (breath-averaged)          {float(v):>12.1f}   {int(v.view(np.uint32)):#010x}')
print()
print('nan_value = -1e30  ->  catches 0xFF7FC99E but not 0xC47A0000')

idx  0  airway_pressure         -3.400e+38   0xff7fc99e
idx  1  flow                    -3.400e+38   0xff7fc99e
idx  2  volume                  -3.400e+38   0xff7fc99e
idx  3  co2_pct                 -3.400e+38   0xff7fc99e
idx  4  co2_kpa                 -3.400e+38   0xff7fc99e
idx  5  co2_mmhg                -3.400e+38   0xff7fc99e

idx  6  (breath-averaged)               -1000.0   0xc47a0000
idx  7  (breath-averaged)               -1000.0   0xc47a0000

nan_value = -1e30  ->  catches 0xFF7FC99E but not 0xC47A0000



### MEDIBUS idx 51-57 sentinels: with and without PressurePod

**Without PressurePod** (Healthy Lung):
- idx 54-57 (Pod channels): `0xFF7FC99E` ≈ -3.4e38 — hardware not connected
- idx 52 (Plow) and idx 53 (Tlow): `-1000.0` — no data computed yet 
- idx 51 (PHigh): `-1000.0` — no data computed yet 

**With PressurePod** (patient01/02):
- idx 54-57: real pressure values (~14 mbar Paw, ~15 mbar Pes, ~-1.3 mbar Ptp, ~0.04 mbar Pgas)
- idx 52 (Plow) and idx 53 (Tlow): `-1000.0` — conventional ventilation (same sentinel as without Pod)
- idx 51 (PHigh): `-1000.0` — no data computed at frame 0 (first frame only; real values appear later)

In [10]:
from fasteit.dtypes import MEDIBUS_EXT_FIELDS

fb = np.memmap(BASE_FILE, dtype=FRAME_EXT_DTYPE, mode='r')
md = fb[0]['medibus_data']

print('=== Healthy Lung: idx 51-57 at frame 0 ===')
for i in range(51, 58):
    name, unit, _ = MEDIBUS_EXT_FIELDS[i]
    v = md[i]
    bits = int(v.view(np.uint32))
    sentinel = '(MEDIBUS/Pod not connected)' if bits == 0xFF7FC99E else '(No data calculated yet)'
    print(f'  idx {i}  {name:<28}  {float(v):>12.4f}   {bits:#010x}  {sentinel}')

=== Healthy Lung: idx 51-57 at frame 0 ===
  idx 51  high_pressure                   -1000.0000   0xc47a0000  (No data calculated yet)
  idx 52  low_pressure                    -1000.0000   0xc47a0000  (No data calculated yet)
  idx 53  time_at_low_pressure            -1000.0000   0xc47a0000  (No data calculated yet)
  idx 54  airway_pressure_pod           -339999995214436424907732413799364296704.0000   0xff7fc99e  (MEDIBUS/Pod not connected)
  idx 55  esophageal_pressure_pod       -339999995214436424907732413799364296704.0000   0xff7fc99e  (MEDIBUS/Pod not connected)
  idx 56  transpulmonary_pressure_pod   -339999995214436424907732413799364296704.0000   0xff7fc99e  (MEDIBUS/Pod not connected)
  idx 57  gastric_pressure_pod          -339999995214436424907732413799364296704.0000   0xff7fc99e  (MEDIBUS/Pod not connected)


### With vs without PressurePod — sentinel comparison

| idx | Field | With PressurePod (patient01) | Without PressurePod (Healthy Lung) |
|-----|-------|------------------------------|-------------------------------------|
| 51 | `high_pressure` (PHigh) | -1000.0 (no breath yet) | -1000.0 (non-BiLevel mode) |
| 52 | `low_pressure` (Plow) | -1000.0 (non-BiLevel mode) | -1000.0 (non-BiLevel mode) |
| 53 | `time_at_low_pressure` (Tlow) | -1000.0 (non-BiLevel mode) | -1000.0 (non-BiLevel mode) |
| 54 | `airway_pressure_pod` | **14.00 mbar** (real) | -3.4e38 (`0xFF7FC99E`) |
| 55 | `esophageal_pressure_pod` | **15.32 mbar** (real) | -3.4e38 (`0xFF7FC99E`) |
| 56 | `transpulmonary_pressure_pod` | **-1.28 mbar** (real) | -3.4e38 (`0xFF7FC99E`) |
| 57 | `gastric_pressure_pod` | **0.04 mbar** (real) | -3.4e38 (`0xFF7FC99E`) |

**Detecting PressurePod presence**: check `medibus_data[:, 54].view(uint32) != 0xFF7FC99E` on a sample of frames in 4382 bytes-frame file version.

## Summary — `FRAME_EXT_DTYPE` layout (4382 bytes)

| offset | hex | bytes | type | field |
| ------ | --- | ----- | ---- | ----- |
| 0 | 0x0000 | 8 | float64 LE | `ts` |
| 8 | 0x0008 | 4 | float32 LE | `dummy` |
| 12 | 0x000C | 4096 | float32 LE | `pixels[32,32]` |
| 4108 | 0x100C | 4 | int32 LE | `min_max_flag` |
| 4112 | 0x1010 | 4 | int32 LE | `event_marker` |
| 4116 | 0x1014 | 30 | bytes | `event_text` |
| 4146 | 0x1032 | 4 | int32 LE | `timing_error` |
| 4150 | 0x1036 | 208 | float32 LE | `medibus_data[0..51]` (52 fields; last = idx 51 = **PHigh** in EXT) |
| 4358 | 0x1106 | 4 | float32 LE | `medibus_data[52]` = Plow |
| 4362 | 0x110A | 4 | float32 LE | `medibus_data[53]` = Tlow |
| 4366 | 0x110E | 4 | float32 LE | `medibus_data[54]` = ~Paw (Pod) |
| 4370 | 0x1112 | 4 | float32 LE | `medibus_data[55]` = ~Pes (Pod) |
| 4374 | 0x1116 | 4 | float32 LE | `medibus_data[56]` = ~Ptp (Pod) |
| 4378 | 0x111A | 4 | float32 LE | `medibus_data[57]` = ~Pgas (Pod) |
| 4382 | | | | end of frame |

**BASE format** (4358 bytes, `FRAME_BASE_DTYPE`) is identical except:
- `medibus_data` ends at idx 51 (no idx 52-57)
- `medibus_data[51]` = **Tlow** (not PHigh — these are different fields at the same index!)
